# BloomML R4 — heavy CUDA physics stress (2 × Tesla T4)

**Research only.** A stress test of numerical conservation, plausible limits and CPU parity. Synthetic, uncalibrated scenarios are **not** evidence that predictions match a real growbox. The simulation backend uses **both CUDA devices**, not a matrix smoke test.


In [ ]:
from pathlib import Path
import subprocess, sys, json
SOURCE = Path('/kaggle/working/bloomml-r4-source')
BRANCH = 'research/simulator-fidelity-kaggle-20261103'
assert not SOURCE.exists(), 'Fresh Kaggle version required for reproducible git clone'
subprocess.run(['git','clone','--branch',BRANCH,'--single-branch','https://github.com/MichalMatu/bloomml.git',str(SOURCE)],check=True)
sha = subprocess.check_output(['git','-C',str(SOURCE),'rev-parse','HEAD'],text=True).strip()
print('RESEARCH_CODE_COMMIT',sha)
sys.path.insert(0,str(SOURCE))


In [ ]:
import torch
from tools.ml.kaggle_r4_stress_runner import run
assert torch.cuda.device_count() >= 2, 'Both Tesla T4 are required; no CPU fallback'
OUTPUT = Path('/kaggle/working/bloomml-r4')
manifest = run(source_dir=SOURCE,output_dir=OUTPUT,seed=1729,
               scenarios_per_gpu=4096,ticks=180,dt_s=10.,pilot_per_gpu=24)
print('SCENARIO_TICKS', manifest['total_scenario_ticks'])
print('NUMERICAL_BOUND_FAILURES',manifest['failed_physical_bounds'])
for item in manifest['devices']:
    print(item['device'],item['gpu_name'],item['total_euler_substeps'],
          item['seconds'],item['bounds'])
print('MANIFEST',OUTPUT/'r4_stress_manifest.json')


In [ ]:
print('Data provenance:',manifest['source_commit'])
print('NUMERICAL physics plausibility only:', not manifest['physical_validation_with_real_data'])
print('Do not publish any real SD telemetry or credentials.')
